# FIRMS fire progression from an existing GeoPackage

This notebook starts with a **previously downloaded FIRMS active-fire point layer** and derives approximate fire progression through time.

It does **not** download FIRMS data. The input is an existing `.gpkg`.

## Outputs

A new GeoPackage is created containing:

- `Alerts_Clustered` — original fire-alert points plus an estimated `fire_id`
- `Progression` — cumulative estimated detection-footprint polygons through time
- `Interval_Growth` — area added since the previous time step
- `Estimated_Front` — approximate newly expanded outer edge

## Important interpretation

These polygons are **not observed wildfire perimeters**.

They are estimates derived from the spatial and temporal pattern of FIRMS thermal detections. Satellite overpass timing, clouds, smoke, detection sensitivity, geolocation uncertainty, and gaps between detections can all affect the result.

The workflow is therefore best treated as a **fire-progression proxy / exploratory product**.

The raw FIRMS points are retained so every derived result can be compared with its source observations.


## Input / output convention

Set `INPUT_GPKG` to any FIRMS GeoPackage.

The progression GeoPackage is written automatically in the **same folder**,
by prefixing the original filename with `Progression_`.

Example:

```text
FIRMS_ALB-WF001_2026-08-13_to_2026-08-24_fire_alerts.gpkg
→
Progression_FIRMS_ALB-WF001_2026-08-13_to_2026-08-24_fire_alerts.gpkg
```


## 1. Install packages

CDSE may already contain these packages. If the imports in the next cell work, this installation cell can be skipped.

In [ ]:
%pip install -q geopandas pandas numpy shapely scikit-learn pyogrio matplotlib

## 2. Imports

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import pyogrio

from sklearn.neighbors import KDTree
from shapely.ops import unary_union

print("Imports successful.")

## 3. User settings

The defaults below are deliberately exposed so that you can experiment.

### Event clustering parameters

`EVENT_LINK_M`

Maximum spatial distance used when deciding whether two detections may belong to the same fire event.

`EVENT_LINK_HOURS`

Maximum time difference between spatially close detections before they are treated as unrelated.

The algorithm builds connected components, so a spreading fire can grow beyond `EVENT_LINK_M` over several observations as long as consecutive observations remain connected.

### Progression parameters

`TIME_STEP`

Use:

```python
TIME_STEP = "1D"
```

for daily progression, or:

```python
TIME_STEP = "12h"
```

for two progression states per day.

`POINT_BUFFER_M`

Each FIRMS point is expanded into a small circular support area before unioning. This is **not intended to reproduce the true satellite pixel geometry**; it simply prevents the progression model from treating each detection as an infinitely small point.

`GAP_CLOSE_M`

A small morphological closing distance that can connect nearby buffered detections and remove narrow gaps.

Set it to `0` if you want only the direct union of point buffers.

### Recommended starting point

For a compact wildfire AOI, start with the defaults and then visually compare the results with the raw points.

In [ ]:
# ============================================================
# USER SETTINGS
# ============================================================

# ------------------------------------------------------------
# INPUT / OUTPUT FILE
# ------------------------------------------------------------

# Point this to the FIRMS GeoPackage you want to process.
INPUT_FOLDER = Path(
    r"../FIRMS/ALB-WF001/outputs"
)

INPUT_FILENAME = "Fire_Alerts_Firms_ALB-WF001_2026-08-13_to_2026-08-24.gpkg"

INPUT_GPKG = INPUT_FOLDER / INPUT_FILENAME

# The progression file is written beside the input file.
# Prefix the original input filename with "Progression_".
OUTPUT_GPKG = INPUT_GPKG.with_name(
    f"Progression_{INPUT_GPKG.name}"
)

print("Input GPKG  :", INPUT_GPKG)
print("Output GPKG :", OUTPUT_GPKG)

# Set to None to let the notebook choose a likely point layer.
INPUT_LAYER = "fire_alerts"

OVERWRITE_OUTPUT = True

# ------------------------------------------------------------
# INPUT FILTERING
# ------------------------------------------------------------

# Recommended for progression work.
# If sensor_name exists, rows containing "VIIRS" are retained.
# Otherwise the notebook checks the instrument field.
#USE_VIIRS_ONLY = True
USE_VIIRS_ONLY = False

# ------------------------------------------------------------
# FIRE-EVENT CLUSTERING
# ------------------------------------------------------------

EVENT_LINK_M = 2500
EVENT_LINK_HOURS = 36

# Events smaller than this are retained in Alerts_Clustered,
# but no progression polygon is generated for them.
MIN_EVENT_POINTS = 3

# ------------------------------------------------------------
# PROGRESSION
# ------------------------------------------------------------

# Examples: "1D", "12h", "6h"
TIME_STEP = "1D"

# Approximate support radius around each detection
POINT_BUFFER_M = 250

# Connect small gaps between nearby detection buffers.
# Set to 0 to disable.
GAP_CLOSE_M = 250

# Used to suppress old boundary segments when estimating
# the newly expanded active front.
FRONT_EXCLUSION_M = 75

# ============================================================

## 5. Select and read the fire-alert point layer

If `INPUT_LAYER = None`, the notebook looks for a layer name containing words such as `alert`, `fire`, or `firm`.

If it cannot find one, it uses the first point-like layer in the GeoPackage.

In [ ]:
if INPUT_LAYER is None:

    layer_names = layers["layer"].tolist()

    likely = [
        name for name in layer_names
        if any(
            word in name.lower()
            for word in ["alert", "fire", "firm"]
        )
    ]

    if likely:
        INPUT_LAYER = likely[0]
    else:
        point_layers = layers[
            layers["geometry_type"]
            .astype(str)
            .str.contains("Point", case=False, na=False)
        ]["layer"].tolist()

        if not point_layers:
            raise ValueError(
                "No point layer was found in the GeoPackage. "
                "Set INPUT_LAYER manually if needed."
            )

        INPUT_LAYER = point_layers[0]

print("Using input layer:", INPUT_LAYER)

alerts = gpd.read_file(
    INPUT_GPKG,
    layer=INPUT_LAYER,
)

print(f"Read {len(alerts):,} features.")
print("CRS:", alerts.crs)

alerts.head()

## 6. Prepare the acquisition timestamp

The preferred field is `acq_datetime`, created by the earlier FIRMS notebook.

If it is not present, this notebook reconstructs UTC acquisition time from the original FIRMS:

- `acq_date`
- `acq_time`

The original fields are left unchanged.

In [ ]:
if "acq_datetime" in alerts.columns:

    alerts["_time_utc"] = pd.to_datetime(
        alerts["acq_datetime"],
        errors="coerce",
        utc=True,
    )

elif {"acq_date", "acq_time"}.issubset(alerts.columns):

    acq_time_str = (
        alerts["acq_time"]
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.zfill(4)
    )

    alerts["_time_utc"] = pd.to_datetime(
        alerts["acq_date"].astype(str)
        + " "
        + acq_time_str.str[:2]
        + ":"
        + acq_time_str.str[2:],
        errors="coerce",
        utc=True,
    )

else:
    raise ValueError(
        "The input needs either an acq_datetime field or "
        "the original FIRMS acq_date + acq_time fields."
    )

before = len(alerts)

alerts = alerts[
    alerts.geometry.notna()
    & alerts["_time_utc"].notna()
].copy()

print(
    f"Removed {before - len(alerts):,} rows with "
    "missing geometry or acquisition time."
)

print("First acquisition:", alerts["_time_utc"].min())
print("Last acquisition: ", alerts["_time_utc"].max())

## 7. Optionally retain VIIRS detections only

VIIRS is usually preferable when constructing this type of progression proxy because mixing sensors with substantially different spatial sampling can make the derived footprint less consistent.

The original GeoPackage is not modified.

In [ ]:
if USE_VIIRS_ONLY:

    before = len(alerts)

    if "sensor_name" in alerts.columns:
        alerts = alerts[
            alerts["sensor_name"]
            .astype(str)
            .str.contains("VIIRS", case=False, na=False)
        ].copy()

    elif "instrument" in alerts.columns:
        alerts = alerts[
            alerts["instrument"]
            .astype(str)
            .str.contains("VIIRS", case=False, na=False)
        ].copy()

    else:
        print(
            "WARNING: no sensor_name or instrument field was found; "
            "the VIIRS-only filter could not be applied."
        )

    print(
        f"VIIRS filter: {before:,} -> {len(alerts):,} detections"
    )

if alerts.empty:
    raise RuntimeError("No detections remain after filtering.")

## 8. Reproject to a local metric CRS

Distances, buffers, and areas should not be calculated directly in longitude/latitude.

GeoPandas estimates an appropriate UTM CRS from the input detections. All progression calculations are performed in this metric CRS.

The final layers are transformed back to the original input CRS before export.

In [ ]:
if alerts.crs is None:
    raise ValueError(
        "The input layer has no CRS. Assign the correct CRS before "
        "running progression analysis."
    )

original_crs = alerts.crs
metric_crs = alerts.estimate_utm_crs()

if metric_crs is None:
    raise RuntimeError(
        "Could not estimate a metric UTM CRS for the detections."
    )

alerts_m = alerts.to_crs(metric_crs).copy()

print("Original CRS:", original_crs)
print("Analysis CRS:", metric_crs)

## 9. Group detections into fire events

This step creates a simple **spatio-temporal connected-component model**.

Two detections are allowed to connect when:

1. they are within `EVENT_LINK_M` metres; and
2. their acquisition times differ by no more than `EVENT_LINK_HOURS`.

Connections are transitive:

```text
A ---- B ---- C
```

If A connects to B and B connects to C, all three belong to the same `fire_id`, even if A and C are farther apart.

That is useful for a moving/spreading fire, but it also means the thresholds matter. Closely spaced independent fires can be merged if the values are too permissive.

In [ ]:
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra = self.find(a)
        rb = self.find(b)

        if ra == rb:
            return

        if self.rank[ra] < self.rank[rb]:
            self.parent[ra] = rb
        elif self.rank[ra] > self.rank[rb]:
            self.parent[rb] = ra
        else:
            self.parent[rb] = ra
            self.rank[ra] += 1


def assign_fire_events(
    gdf,
    spatial_link_m,
    temporal_link_hours,
):
    gdf = gdf.sort_values("_time_utc").copy()
    gdf = gdf.reset_index(drop=False).rename(
        columns={"index": "_original_index"}
    )

    xy = np.column_stack(
        [
            gdf.geometry.x.to_numpy(),
            gdf.geometry.y.to_numpy(),
        ]
    )

    time_hours = (
        gdf["_time_utc"].astype("int64").to_numpy()
        / 3_600_000_000_000
    )

    tree = KDTree(xy)
    neighbor_lists = tree.query_radius(
        xy,
        r=spatial_link_m,
    )

    uf = UnionFind(len(gdf))

    for i, neighbors in enumerate(neighbor_lists):

        for j in neighbors:

            j = int(j)

            if j <= i:
                continue

            time_difference = abs(
                time_hours[j] - time_hours[i]
            )

            if time_difference <= temporal_link_hours:
                uf.union(i, j)

    roots = [uf.find(i) for i in range(len(gdf))]
    gdf["_component"] = roots

    # Number events according to their first observation time.
    component_order = (
        gdf.groupby("_component")["_time_utc"]
        .min()
        .sort_values()
        .index
        .tolist()
    )

    component_to_fire = {
        component: f"F{number:04d}"
        for number, component in enumerate(
            component_order,
            start=1,
        )
    }

    gdf["fire_id"] = gdf["_component"].map(
        component_to_fire
    )

    return gdf


alerts_m = assign_fire_events(
    alerts_m,
    spatial_link_m=EVENT_LINK_M,
    temporal_link_hours=EVENT_LINK_HOURS,
)

event_counts = (
    alerts_m.groupby("fire_id")
    .size()
    .sort_values(ascending=False)
    .rename("detections")
    .reset_index()
)

print("Estimated fire events:", len(event_counts))
event_counts.head(20)

## 10. Review event sizes

Small clusters may be isolated hot pixels, unrelated thermal anomalies, or simply fires with too few satellite observations to produce a useful progression polygon.

The notebook retains all points in `Alerts_Clustered`, but only events containing at least `MIN_EVENT_POINTS` detections are used for progression.

In [ ]:
eligible_fire_ids = event_counts.loc[
    event_counts["detections"] >= MIN_EVENT_POINTS,
    "fire_id",
].tolist()

print(
    f"{len(eligible_fire_ids)} of {len(event_counts)} events "
    f"have at least {MIN_EVENT_POINTS} detections."
)

event_counts

## 11. Define the estimated footprint

For each time step, all detections observed up to the end of that interval are accumulated.

Each point is buffered by `POINT_BUFFER_M`, and those support areas are unioned.

If `GAP_CLOSE_M > 0`, a small outward/inward buffering operation closes narrow gaps between nearby support areas.

This produces a **cumulative detection footprint**, not a mapped burned-area perimeter.

In [ ]:
def build_detection_footprint(
    point_geometries,
    point_buffer_m=250,
    gap_close_m=0,
):
    geoms = [
        geom.buffer(point_buffer_m)
        for geom in point_geometries
        if geom is not None and not geom.is_empty
    ]

    if not geoms:
        return None

    footprint = unary_union(geoms)

    if gap_close_m > 0:
        footprint = (
            footprint
            .buffer(gap_close_m)
            .buffer(-gap_close_m)
        )

    if footprint.is_empty:
        return None

    return footprint

## 12. Build cumulative progression, growth, and estimated fronts

For each eligible `fire_id`, the notebook creates regular time intervals.

At each step:

- `Progression` = cumulative footprint using all detections observed so far
- `Interval_Growth` = current cumulative footprint minus the previous footprint
- `Estimated_Front` = portions of the current outer boundary that are not close to the previous footprint

Useful attributes include:

- `fire_id`
- `step_start`
- `step_end`
- `detections_step`
- `detections_total`
- `area_ha`
- `growth_ha`
- `max_frp_step`
- `mean_frp_step`
- `sensors_step`

A zero-detection interval is retained in the progression sequence, so a daily timeline remains continuous even if there was no satellite detection on a particular day.

In [ ]:
progression_rows = []
growth_rows = []
front_rows = []

time_offset = pd.tseries.frequencies.to_offset(TIME_STEP)

for fire_id in eligible_fire_ids:

    event = alerts_m[
        alerts_m["fire_id"] == fire_id
    ].sort_values("_time_utc").copy()

    event_start = event["_time_utc"].min().floor(TIME_STEP)
    event_last = event["_time_utc"].max().floor(TIME_STEP)

    step_starts = pd.date_range(
        start=event_start,
        end=event_last,
        freq=TIME_STEP,
        tz="UTC",
    )

    previous_footprint = None

    for step_start in step_starts:

        step_end = step_start + time_offset

        step_points = event[
            (event["_time_utc"] >= step_start)
            & (event["_time_utc"] < step_end)
        ]

        cumulative = event[
            event["_time_utc"] < step_end
        ]

        current_footprint = build_detection_footprint(
            cumulative.geometry,
            point_buffer_m=POINT_BUFFER_M,
            gap_close_m=GAP_CLOSE_M,
        )

        if current_footprint is None:
            continue

        if previous_footprint is None:
            growth = current_footprint
            front = current_footprint.boundary
        else:
            growth = current_footprint.difference(
                previous_footprint
            )

            # Keep newly expanded outer-boundary segments.
            front = current_footprint.boundary.difference(
                previous_footprint.buffer(
                    FRONT_EXCLUSION_M
                )
            )

        area_ha = current_footprint.area / 10_000
        growth_ha = (
            0.0
            if growth is None or growth.is_empty
            else growth.area / 10_000
        )

        if "frp" in step_points.columns and not step_points.empty:
            frp_values = pd.to_numeric(
                step_points["frp"],
                errors="coerce",
            )
            max_frp_step = frp_values.max()
            mean_frp_step = frp_values.mean()
        else:
            max_frp_step = np.nan
            mean_frp_step = np.nan

        if "sensor_name" in step_points.columns:
            sensors = sorted(
                step_points["sensor_name"]
                .dropna()
                .astype(str)
                .unique()
                .tolist()
            )
            sensors_step = ", ".join(sensors)
        elif "satellite" in step_points.columns:
            sensors = sorted(
                step_points["satellite"]
                .dropna()
                .astype(str)
                .unique()
                .tolist()
            )
            sensors_step = ", ".join(sensors)
        else:
            sensors_step = ""

        common = {
            "fire_id": fire_id,
            "step_start": step_start,
            "step_end": step_end,
            "detections_step": len(step_points),
            "detections_total": len(cumulative),
            "area_ha": area_ha,
            "growth_ha": growth_ha,
            "max_frp_step": max_frp_step,
            "mean_frp_step": mean_frp_step,
            "sensors_step": sensors_step,
        }

        progression_rows.append(
            {
                **common,
                "geometry": current_footprint,
            }
        )

        if growth is not None and not growth.is_empty:
            growth_rows.append(
                {
                    **common,
                    "geometry": growth,
                }
            )

        if front is not None and not front.is_empty:
            front_rows.append(
                {
                    **common,
                    "geometry": front,
                }
            )

        previous_footprint = current_footprint


progression_m = gpd.GeoDataFrame(
    progression_rows,
    geometry="geometry",
    crs=metric_crs,
)

growth_m = gpd.GeoDataFrame(
    growth_rows,
    geometry="geometry",
    crs=metric_crs,
)

front_m = gpd.GeoDataFrame(
    front_rows,
    geometry="geometry",
    crs=metric_crs,
)

print("Progression states:", len(progression_m))
print("Growth polygons:   ", len(growth_m))
print("Front states:      ", len(front_m))

## 13. Add progression time-step information back to the points

This makes the raw clustered detections convenient to animate alongside the derived polygons in QGIS.

In [ ]:
alerts_m["progression_step"] = (
    alerts_m["_time_utc"].dt.floor(TIME_STEP)
)

event_size_map = (
    alerts_m.groupby("fire_id")
    .size()
    .to_dict()
)

alerts_m["event_detections"] = (
    alerts_m["fire_id"].map(event_size_map)
)

alerts_clustered = alerts_m.drop(
    columns=["_component"],
    errors="ignore",
).to_crs(original_crs)

alerts_clustered.head()

## 14. Inspect the progression table

For daily analysis, each row represents the cumulative estimated footprint at the end of one day.

`growth_ha` is the additional support area compared with the preceding interval.

In [ ]:
progression_m[
    [
        "fire_id",
        "step_start",
        "step_end",
        "detections_step",
        "detections_total",
        "area_ha",
        "growth_ha",
        "max_frp_step",
        "sensors_step",
    ]
].head(30)

## 15. Quick visual check

This plots one fire event so you can compare:

- raw detection points
- cumulative progression boundaries

Change `FIRE_TO_PLOT` to inspect another event.

In [ ]:
if eligible_fire_ids:

    FIRE_TO_PLOT = eligible_fire_ids[0]

    event_points = alerts_m[
        alerts_m["fire_id"] == FIRE_TO_PLOT
    ].copy()

    event_progression = progression_m[
        progression_m["fire_id"] == FIRE_TO_PLOT
    ].copy()

    # Convert acquisition time to elapsed hours for plotting
    first_time = event_points["_time_utc"].min()

    event_points["hours_since_start"] = (
        event_points["_time_utc"] - first_time
    ).dt.total_seconds() / 3600

    fig, ax = plt.subplots(figsize=(9, 9))

    # Estimated cumulative progression boundaries
    if not event_progression.empty:
        event_progression.boundary.plot(
            ax=ax,
            linewidth=1.5,
        )

    # Raw FIRMS detections
    event_points.plot(
        ax=ax,
        column="hours_since_start",
        markersize=20,
        legend=True,
        legend_kwds={
            "label": "Hours since first FIRMS detection"
        },
    )

    ax.set_title(
        f"{FIRE_TO_PLOT}: FIRMS detections and estimated progression"
    )

    ax.set_axis_off()

    plt.show()

else:
    print("No eligible fire event is available to plot.")

## 16. Reproject derived layers back to the input CRS

Areas were calculated in the metric analysis CRS. The exported geometries are returned to the CRS of your original FIRMS GeoPackage so that the layers line up naturally in QGIS.

In [ ]:
progression = progression_m.to_crs(original_crs)
growth = growth_m.to_crs(original_crs)
front = front_m.to_crs(original_crs)

print("Export CRS:", original_crs)

## 17. Export the progression GeoPackage

The source GeoPackage is left untouched.

The new GeoPackage contains four vector layers:

```text
Alerts_Clustered
Progression
Interval_Growth
Estimated_Front
```

If `OVERWRITE_OUTPUT = True`, an existing output GeoPackage with the same name is deleted before writing.

In [ ]:
if OVERWRITE_OUTPUT and OUTPUT_GPKG.exists():
    OUTPUT_GPKG.unlink()

alerts_clustered.to_file(
    OUTPUT_GPKG,
    layer="Alerts_Clustered",
    driver="GPKG",
)

if not progression.empty:
    progression.to_file(
        OUTPUT_GPKG,
        layer="Progression",
        driver="GPKG",
    )

if not growth.empty:
    growth.to_file(
        OUTPUT_GPKG,
        layer="Interval_Growth",
        driver="GPKG",
    )

if not front.empty:
    front.to_file(
        OUTPUT_GPKG,
        layer="Estimated_Front",
        driver="GPKG",
    )

print("Finished.")
print("Output:", OUTPUT_GPKG)
print()
print("Layers:")
print("  Alerts_Clustered :", len(alerts_clustered))
print("  Progression      :", len(progression))
print("  Interval_Growth  :", len(growth))
print("  Estimated_Front  :", len(front))

## 18. QGIS temporal setup

### Raw detections

For `Alerts_Clustered`, use:

```text
progression_step
```

or the original:

```text
acq_datetime
```

as the temporal field.

### Progression polygons

For `Progression`, use:

```text
step_start
```

and optionally `step_end` as the end datetime.

### Growth polygons

`Interval_Growth` shows only the newly added estimated area for each interval.

### Front lines

`Estimated_Front` is useful as a visual indicator of where the derived footprint expanded.

A useful QGIS layout is:

```text
Estimated_Front      top
fire alerts
Interval_Growth
Progression          bottom
satellite basemap
```

Remember that these are FIRMS-derived estimates rather than validated wildfire perimeter observations.

## 19. Parameters worth testing

There is no universal parameter set for every fire.

### If separate nearby fires are being merged

Reduce one or both:

```python
EVENT_LINK_M = 1500
EVENT_LINK_HOURS = 24
```

### If one known fire is being split into many events

Increase them moderately:

```python
EVENT_LINK_M = 3500
EVENT_LINK_HOURS = 48
```

### If progression polygons are too fragmented

Increase:

```python
GAP_CLOSE_M = 400
```

### If polygons are too broad

Reduce:

```python
POINT_BUFFER_M = 180
GAP_CLOSE_M = 100
```

### For more temporal detail

Try:

```python
TIME_STEP = "12h"
```

This is often useful when detections from multiple VIIRS platforms occur at different overpass times.

Always compare the derived polygons and front lines against the raw FIRMS detections before interpreting them.

## 20. Possible next improvement

This progression model intentionally uses a transparent and relatively simple **buffered-detection footprint**.

A more advanced second version could add:

- concave/alpha-shape envelopes
- sensor-dependent footprint sizes
- filtering by FIRMS confidence
- detection persistence rules
- separate active/inactive front classification
- Sentinel-2 or Landsat burned-area constraints
- comparison against EFFIS perimeters
- fire-event merging/splitting review tools

This is a rather simple model that makes it easier to understand where every derived polygon came from.